# YOLO26s Detector Training on Roboflow

AdamW + cosine LR, early stopping, periodic checkpoints, and metrics export.

In [ ]:
# %pip install ultralytics roboflow

In [ ]:
from __future__ import annotations
import json, os, shutil
from datetime import datetime
from pathlib import Path
from typing import Any
from ultralytics import YOLO

In [ ]:
MODEL_NAME='yolo26s.pt'
EPOCHS=100
IMGSZ=640
BATCH=16
WORKERS=8
DEVICE=None
OPTIMIZER='AdamW'
LR0=1e-3
LRF=1e-2
WEIGHT_DECAY=5e-4
COS_LR=True
PATIENCE=20
SAVE_PERIOD=5
OUTPUT_ROOT=Path('runs')/'yolo26s_roboflow'
RUN_NAME=f"train_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
DATA_YAML=None
ROBOFLOW_API_KEY=os.getenv('ROBOFLOW_API_KEY', None)
ROBOFLOW_WORKSPACE=None
ROBOFLOW_PROJECT=None
ROBOFLOW_VERSION=None
ROBOFLOW_FORMAT='yolov8'
DATASET_CACHE_DIR=Path('datasets')/'roboflow'
FORCE_DOWNLOAD=False

In [ ]:
def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict): return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [to_jsonable(v) for v in obj]
    if isinstance(obj, Path): return str(obj)
    if hasattr(obj, 'item'):
        try: return obj.item()
        except Exception: return str(obj)
    return obj

def resolve_data_yaml(data_yaml, api_key, workspace, project, version, export_format='yolov8', dataset_cache_dir=Path('datasets')/'roboflow', force_download=False):
    if data_yaml is not None:
        p = Path(data_yaml).resolve()
        if not p.exists(): raise FileNotFoundError(f'Provided data.yaml not found: {p}')
        return p
    missing=[]
    if not api_key: missing.append('ROBOFLOW_API_KEY')
    if not workspace: missing.append('ROBOFLOW_WORKSPACE')
    if not project: missing.append('ROBOFLOW_PROJECT')
    if version is None: missing.append('ROBOFLOW_VERSION')
    if missing: raise ValueError('Missing dataset config: ' + ', '.join(missing))
    from roboflow import Roboflow
    target_dir=(dataset_cache_dir/workspace/project/f'v{version}').resolve()
    target_yaml=target_dir/'data.yaml'
    if target_yaml.exists() and not force_download: return target_yaml
    target_dir.mkdir(parents=True, exist_ok=True)
    rf=Roboflow(api_key=api_key)
    ds=rf.workspace(workspace).project(project).version(version).download(export_format, location=str(target_dir))
    y=Path(ds.location)/'data.yaml'
    if not y.exists(): raise FileNotFoundError(f'Roboflow download missing data.yaml: {y}')
    return y.resolve()

def export_metrics(run_dir: Path, train_result: Any, val_result: Any) -> Path:
    m=run_dir/'metrics'; m.mkdir(parents=True, exist_ok=True)
    summary={
      'timestamp_utc': datetime.utcnow().isoformat(timespec='seconds')+'Z',
      'run_dir': str(run_dir),
      'train_result': to_jsonable(getattr(train_result,'results_dict',{})),
      'val_result': to_jsonable(getattr(val_result,'results_dict',{})),
      'best_checkpoint': str(run_dir/'weights'/'best.pt'),
      'last_checkpoint': str(run_dir/'weights'/'last.pt')
    }
    (m/'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    for a in ['results.csv','results.png','args.yaml','confusion_matrix.png']:
        s=run_dir/a
        if s.exists(): shutil.copy2(s, m/s.name)
    return m

In [ ]:
data_yaml_path = resolve_data_yaml(
    data_yaml=DATA_YAML,
    api_key=ROBOFLOW_API_KEY,
    workspace=ROBOFLOW_WORKSPACE,
    project=ROBOFLOW_PROJECT,
    version=ROBOFLOW_VERSION,
    export_format=ROBOFLOW_FORMAT,
    dataset_cache_dir=DATASET_CACHE_DIR,
    force_download=FORCE_DOWNLOAD,
)
data_yaml_path

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
model = YOLO(MODEL_NAME)
train_kwargs = {
 'data': str(data_yaml_path), 'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH,
 'workers': WORKERS, 'device': DEVICE, 'optimizer': OPTIMIZER,
 'lr0': LR0, 'lrf': LRF, 'weight_decay': WEIGHT_DECAY, 'cos_lr': COS_LR,
 'patience': PATIENCE, 'save': True, 'save_period': SAVE_PERIOD,
 'project': str(OUTPUT_ROOT.resolve()), 'name': RUN_NAME, 'exist_ok': True,
}
print(json.dumps(to_jsonable(train_kwargs), indent=2))
train_result = model.train(**train_kwargs)
run_dir = Path(getattr(train_result, 'save_dir', OUTPUT_ROOT / RUN_NAME))
print('Run dir:', run_dir)

In [ ]:
val_result = model.val(data=str(data_yaml_path), split='val')
metrics_dir = export_metrics(run_dir, train_result, val_result)
print('Metrics folder:', metrics_dir)
print('Best weights:', run_dir / 'weights' / 'best.pt')
print('Last weights:', run_dir / 'weights' / 'last.pt')